# BQNT option-price history and rolling Z-score dashboard — revision 2

**This revision changes the price source from bid/ask midpoint to `PX_LAST`.** Latest and historical prices use the same field; the dashboard does not mix a current midpoint with historical last prices.

The previous output's **zero prior observations** means zero usable prior midpoint observations, not proof that the BQL response contained only one row. The diagnostic below separates returned rows from valid prices and can compare `PX_LAST`, `PX_BID`, and `PX_ASK`.

**Two distinct series are supported, with explicit labels:**

- **Default: same contract.** Select today's near-ATM matched call/put pair, then query each contract's own history. This fixes the history retrieval path but is not a daily rolled ATM series.
- **With actual historical chain snapshots: daily reselected ATM.** Select a matched pair on each dated snapshot using that date's spot, query the selected identifiers through BQL, and retain exactly one call/put observation per date. No expired identifiers or backdated `options()` parameters are invented. Acquiring historical chain snapshots is not automated in this notebook.

All market-price queries execute in your entitled BQNT session. No demonstration-price mode is included. Live Bloomberg requests and BQNT rendering have not been tested by the author; offline response-processing and calculation tests passed.


## 1. Configuration and calculations
Run the definition cells first. The default baseline is the prior 365 calendar days, excluding the current observation; 200 valid prior observations and a full elapsed year are required. A 740-day fetch provides warm-up where that much history exists.

In [ ]:
"""Bloomberg BQNT-only S&P 500 option PX_LAST history dashboard — revision 2.

No synthetic-data mode and no CSV price loader. Price source: BQL PX_LAST.
Run in an entitled Bloomberg BQuant notebook. Live BQL execution is untested here.

Default: select a CURRENT listed call/put pair and use its OWN historical PX_LAST.
Optional: pass REAL dated chain_snapshots to launch_dashboard for daily reselection.
The optional path requires historical chain identifiers; it does NOT backdate
options(), invent expired tickers, or claim that today's chain covers the past.
Nearest-listed maturity is an approximation, not exact constant maturity.
"""
from __future__ import annotations

from functools import reduce
from html import escape
from typing import Callable, Optional

import numpy as np
import pandas as pd

MODE = "BQNT"  # Informational only: Bloomberg is the only data path.
INDEX = "SPX Index"
TARGET_DTE = 30
EXPIRY_TOLERANCE_DAYS = 14
MIN_DTE = 7
WINDOW_DAYS = 365
MIN_OBSERVATIONS = 200
REQUIRE_FULL_WINDOW = True
HISTORY_DAYS = 740  # Warm-up for a chart of rolling 365-day scores.
PRICE_FIELD = "PX_LAST"
BATCH_SIZE = 40
MAX_STOCKS = None  # None = every current constituent; 5 = connection test.

REQUIRED = ["date", "ticker", "spot", "call_px", "put_px"]
METRICS = {"Call Z": "call", "Put Z": "put", "Spread Z": "spread"}


def validate_history(raw: pd.DataFrame) -> pd.DataFrame:
    """Validate daily observations for a defined call/put selection rule.

    Prices must be in the same currency and quoted premium units; dates must be timezone-naive US
    market-date labels. Missing observations are allowed but never filled.
    The caller must verify contract selection and price conventions;
    the five-column schema alone cannot establish these properties.
    """
    if not isinstance(raw, pd.DataFrame) or raw.empty:
        raise ValueError("The history loader must return a nonempty DataFrame.")
    missing = set(REQUIRED).difference(raw.columns)
    if missing:
        raise ValueError("Missing input columns: " + ", ".join(sorted(missing)))
    d = raw[REQUIRED].copy()
    d["date"] = pd.to_datetime(d["date"], errors="raise")
    if d["date"].isna().any():
        raise ValueError("Dates cannot be missing.")
    if d["date"].dt.tz is not None:
        raise ValueError("Use timezone-naive US market-date labels, not timestamps.")
    d["date"] = d["date"].dt.normalize()
    if d["ticker"].isna().any():
        raise ValueError("Ticker values cannot be missing.")
    d["ticker"] = d["ticker"].astype(str).str.strip()
    if d["ticker"].eq("").any():
        raise ValueError("Ticker values cannot be blank.")
    if d.duplicated(["date", "ticker"]).any():
        raise ValueError("Duplicate date/ticker rows: select a single consistent option series first.")
    for col in ["spot", "call_px", "put_px"]:
        d[col] = pd.to_numeric(d[col], errors="raise").astype(float)
        if np.isinf(d[col]).any():
            raise ValueError(f"Infinite values in {col}.")
        invalid = d[col].le(0) if col == "spot" else d[col].lt(0)
        if invalid.any():
            raise ValueError(f"Invalid negative/zero price in {col}.")
    return d.sort_values(["date", "ticker"]).reset_index(drop=True)


def compute_dashboard(
    raw: pd.DataFrame,
    basis: str = "Raw premium",
    window_days: int = 365,
    min_obs: int = 200,
    asof: Optional[str] = None,
    require_full_window: bool = REQUIRE_FULL_WINDOW,
) -> dict:
    """Calculate per-stock trailing z-scores, never cross-sectional z-scores.

    Baseline at t: dates in [t-window_days, t); today's value is excluded.
    By default a score requires min_obs valid prior observations AND at least
    window_days of elapsed observed history. Disabling require_full_window
    is an explicit available-history statistic, NOT full-year coverage. Standard deviation
    uses ddof=1. A zero or invalid standard deviation produces NaN, not zero.
    This is a 365-day convention, not an exact anniversary in leap years.
    """
    if window_days < 2 or min_obs < 2:
        raise ValueError("window_days and min_obs must both be >= 2.")
    if basis not in {"Raw premium", "Premium / spot"}:
        raise ValueError("Unknown price basis.")
    d = validate_history(raw)
    stamp = pd.Timestamp(asof).normalize() if asof else d["date"].max()
    if stamp.tzinfo is not None:
        raise ValueError("asof must be a timezone-naive market date.")
    d = d.loc[d["date"].le(stamp)].copy()
    if d.empty:
        raise ValueError("There are no observations on or before the requested date.")
    tickers = sorted(d["ticker"].unique())
    dates = pd.DatetimeIndex(sorted(set(d["date"]) | {stamp}), name="date")
    price = {
        col: d.pivot(index="date", columns="ticker", values=col).reindex(
            index=dates, columns=tickers
        )
        for col in ["spot", "call_px", "put_px"]
    }
    call, put = price["call_px"], price["put_px"]
    if basis == "Premium / spot":
        call = 100.0 * call.div(price["spot"])
        put = 100.0 * put.div(price["spot"])
    values = {"call": call, "put": put, "spread": call - put}
    scores, counts, means, stds, statuses = {}, {}, {}, {}, {}
    delta = pd.Timedelta(days=window_days)
    for metric, x in values.items():
        rolling = x.rolling(f"{window_days}D", closed="left", min_periods=2)
        mean, std = rolling.mean(), rolling.std(ddof=1)
        count = x.rolling(f"{window_days}D", closed="left", min_periods=0).count()
        mature = pd.DataFrame(False, index=dates, columns=tickers)
        for ticker in tickers:
            first = x[ticker].first_valid_index()
            if first is not None:
                mature[ticker] = dates >= first + delta
        eligible = count.ge(min_obs) & (mature if require_full_window else True)
        z = ((x - mean) / std.where(std.gt(1e-12))).where(eligible)
        scores[metric], counts[metric] = z, count
        means[metric], stds[metric] = mean, std
        status = pd.Series("OK" if require_full_window else "OK — available history",
                           index=tickers, dtype=object)
        status.loc[~std.loc[stamp].gt(1e-12)] = "Zero/invalid standard deviation"
        status.loc[count.loc[stamp].lt(min_obs)] = f"Fewer than {min_obs} prior observations"
        if require_full_window:
            status.loc[~mature.loc[stamp]] = f"Observed history spans less than {window_days} days"
        status.loc[count.loc[stamp].eq(0)] = "No valid prior prices in window"
        status.loc[x.loc[stamp].isna()] = "No value at the as-of date"
        statuses[metric] = status

    latest = pd.DataFrame(index=pd.Index(tickers, name="Ticker"))
    latest["Spot"] = price["spot"].loc[stamp]
    latest["Call PX_LAST"] = price["call_px"].loc[stamp]
    latest["Put PX_LAST"] = price["put_px"].loc[stamp]
    for label, metric in METRICS.items():
        latest[label] = scores[metric].loc[stamp]
    quote_rows = d.loc[d[["call_px", "put_px"]].notna().any(axis=1)]
    last_quote = quote_rows.groupby("ticker")["date"].max()
    latest["Last option date"] = last_quote.reindex(tickers)
    return dict(
        latest=latest, scores=scores, values=values, price=price,
        counts=counts, means=means, stds=stds, statuses=statuses,
        asof=stamp, window_days=window_days, min_obs=min_obs, basis=basis,
        require_full_window=require_full_window,
    )




## 2. Bloomberg retrieval and history diagnostics
Historical prices are kept on `(ID, DATE)`. Static metadata stays keyed by `ID`; metadata rows are not vertically appended to price rows. The loader never forward-fills missing prices.

In [ ]:

def request_frame(bq, universe, items: dict, history: bool = False) -> pd.DataFrame:
    """Join named BQL results explicitly; reject ambiguous observations."""
    import bql
    response = bq.execute(bql.Request(universe, items))
    if len(response) != len(items):
        raise ValueError("BQL returned a different number of fields than requested.")
    keys = ["ID", "DATE"] if history else ["ID"]
    pieces = []
    for name, result in zip(items, response):
        frame = result.df().reset_index()
        if frame.empty:
            pieces.append(pd.DataFrame(columns=keys + [name]))
            continue
        missing = set(keys + [name]).difference(frame.columns)
        if missing:
            raise ValueError(f"{name}: missing BQL columns {sorted(missing)}; got {list(frame.columns)}")
        if "CURRENCY" in frame and history:
            currencies = set(frame["CURRENCY"].dropna().astype(str).str.upper())
            if currencies - {"USD"}:
                raise ValueError(f"{name}: expected USD prices; got {currencies}.")
        frame = frame[keys + [name]].copy()
        if frame["ID"].isna().any():
            raise ValueError(f"{name}: BQL returned a missing security ID.")
        frame["ID"] = frame["ID"].astype(str)
        if history:
            frame["DATE"] = pd.to_datetime(frame["DATE"], errors="raise")
            if frame["DATE"].isna().any() or frame["DATE"].dt.tz is not None:
                raise ValueError(f"{name}: expected timezone-naive market dates.")
            frame["DATE"] = frame["DATE"].dt.normalize()
            frame[name] = pd.to_numeric(frame[name], errors="raise").replace([np.inf, -np.inf], np.nan)
        if frame.duplicated(keys).any():
            raise ValueError(f"{name}: duplicate {keys}; inspect returned BQL dimensions.")
        pieces.append(frame)
    return reduce(
        lambda a, b: a.merge(b, on=keys, how="outer", validate="one_to_one"),
        pieces,
    )


def select_pair(chain: pd.DataFrame, spot: float, asof: pd.Timestamp,
                target_dte: int, tolerance: int) -> dict:
    """Closest eligible expiry first, then closest strike with one call and put.

    Reject duplicate expiry/strike/side records rather than silently choosing an
    exchange or adjusted contract. Unique pairs still require deliverable review.
    """
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError("No valid underlying price on the common as-of date.")
    c = chain.copy()
    c["Expiry"] = pd.to_datetime(c["Expiry"], errors="coerce").dt.normalize()
    c["Strike"] = pd.to_numeric(c["Strike"], errors="coerce")
    c["Side"] = c["Side"].astype(str).str.strip().str.upper().map(
        {"CALL": "C", "C": "C", "PUT": "P", "P": "P"}
    )
    c["DTE"] = (c["Expiry"] - asof).dt.days
    c = c.loc[c["Side"].notna() & c["Strike"].gt(0)
              & c["DTE"].ge(MIN_DTE)
              & c["DTE"].sub(target_dte).abs().le(tolerance)].copy()
    c = c.loc[~c.duplicated(["Expiry", "Strike", "Side"], keep=False)]
    if c.empty:
        raise ValueError("No unambiguous option pair within the requested expiry band.")
    pairs = c.pivot(index=["Expiry", "Strike", "DTE"], columns="Side", values="ID")
    if not {"C", "P"}.issubset(pairs.columns):
        raise ValueError("No matched call and put in the eligible chain.")
    pairs = pairs.dropna(subset=["C", "P"]).reset_index()
    if pairs.empty:
        raise ValueError("No call and put share an eligible expiry and strike.")
    pairs["expiry_distance"] = pairs["DTE"].sub(target_dte).abs()
    pairs["strike_distance"] = pairs["Strike"].sub(spot).abs()
    p = pairs.sort_values(["expiry_distance", "Expiry", "strike_distance", "Strike"]).iloc[0]
    return {"Expiry": p["Expiry"], "Strike": float(p["Strike"]),
            "DTE": int(p["DTE"]), "Call contract": p["C"], "Put contract": p["P"]}


def market_date(value=None) -> pd.Timestamp:
    """New York market-date label; not a trading-calendar or close-time filter."""
    t = pd.Timestamp(value) if value is not None else pd.Timestamp.now(tz="America/New_York")
    if t.tzinfo is not None:
        t = t.tz_convert("America/New_York").tz_localize(None)
    return t.normalize()


def history_coverage(prices: pd.DataFrame, ids, column="PX_LAST",
                     start=None, end=None) -> pd.DataFrame:
    """Separate returned dates from dates with a finite nonnegative price."""
    records = []
    for sid in ids:
        q = prices.loc[prices["ID"].eq(sid)].copy()
        q[column] = pd.to_numeric(q[column], errors="coerce").astype(float)
        good = q.loc[np.isfinite(q[column]) & q[column].ge(0)]
        records.append({
            "ID": sid, "Requested start": start, "Requested end": end,
            "Returned rows": len(q), "Returned dates": q["DATE"].nunique(),
            "Valid prices": good["DATE"].nunique(),
            "First returned date": q["DATE"].min(), "Last returned date": q["DATE"].max(),
            "First valid date": good["DATE"].min(), "Last valid date": good["DATE"].max(),
        })
    return pd.DataFrame(records).set_index("ID")


def probe_option_history(bq, contract_ids, start=None, end=None,
                         compare_bid_ask=False) -> dict:
    """Small BQNT diagnostic. The primary request follows the supplied PX_LAST syntax.

    Each price field is requested independently. It is therefore possible to see
    that PX_LAST has history even when a bid/ask request fails or returns few prices.
    No missing values are forward filled. Returned dates are NOT trade timestamps.
    """
    end = market_date(end)
    start = market_date(start) if start is not None else end - pd.Timedelta(days=365)
    if start > end:
        raise ValueError("start must be on or before end")
    ids = list(dict.fromkeys(str(x) for x in contract_ids))
    if not ids:
        raise ValueError("At least one verified Bloomberg contract identifier is required.")
    dates = bq.func.range(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    fields = ["PX_LAST", "PX_BID", "PX_ASK"] if compare_bid_ask else ["PX_LAST"]
    coverage, frames, errors = [], {}, []
    for field in fields:
        try:
            item = getattr(bq.data, field.lower())(dates=dates, fill="NA")
            frame = request_frame(bq, ids, {field: item}, history=True)
            frames[field] = frame
            report = history_coverage(frame, ids, field, start, end).reset_index()
            report.insert(1, "Field", field)
            coverage.append(report)
        except Exception as exc:
            errors.append({"Field": field, "Error": str(exc)})
    return {"coverage": pd.concat(coverage, ignore_index=True) if coverage else pd.DataFrame(),
            "frames": frames, "errors": pd.DataFrame(errors)}


def fetch_px_last(bq, ids, start, end, progress=lambda _: None) -> tuple:
    """Retrieve only daily price rows, in batches; never mix static metadata into them."""
    ids = sorted(set(str(x) for x in ids))
    if BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be positive")
    dates = bq.func.range(pd.Timestamp(start).strftime("%Y-%m-%d"),
                         pd.Timestamp(end).strftime("%Y-%m-%d"))
    item = {"PX_LAST": bq.data.px_last(dates=dates, fill="NA")}
    chunks, errors = [], []
    successes = 0
    for i in range(0, len(ids), BATCH_SIZE):
        batch = ids[i:i+BATCH_SIZE]
        progress(f"PX_LAST history: {min(i+BATCH_SIZE, len(ids))}/{len(ids)} contracts")
        try:
            f = request_frame(bq, batch, item, history=True)
            f.loc[f["PX_LAST"].lt(0), "PX_LAST"] = np.nan
            chunks.append(f)
            successes += 1
        except Exception as exc:
            errors.append({"Stage": "PX_LAST history", "Security": ", ".join(batch), "Error": str(exc)})
            if successes == 0 and len(errors) >= 3:
                raise RuntimeError("The first three PX_LAST batches failed: " + str(exc)) from exc
    prices = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=["ID", "DATE", "PX_LAST"])
    if ids and successes == 0:
        raise RuntimeError("No historical price request succeeded: " + str(errors))
    if prices.duplicated(["ID", "DATE"]).any():
        raise ValueError("Duplicate contract/date price rows across batches; inspect security aliases.")
    prices["DATE"] = pd.to_datetime(prices["DATE"])
    prices["PX_LAST"] = pd.to_numeric(prices["PX_LAST"], errors="raise")
    return prices, errors


def fetch_bloomberg_history(bq, progress: Callable[[str], None] = lambda _: None,
                           target_dte: int = TARGET_DTE,
                           tolerance: int = EXPIRY_TOLERANCE_DAYS,
                           max_stocks: Optional[int] = MAX_STOCKS) -> dict:
    """Current matched call/put pairs + each contract's OWN historical PX_LAST.

    Not a historical rolled ATM series. No bid/ask-to-last substitution within a
    series. All historical and latest option values use the same PX_LAST field.
    Availability and BQL requests require validation in the user's BQNT session.
    """
    if target_dte < MIN_DTE or tolerance < 0:
        raise ValueError(f"Target DTE must be >= {MIN_DTE}; tolerance must be >= 0.")
    if max_stocks is not None and max_stocks < 1:
        raise ValueError("MAX_STOCKS must be None or a positive integer.")
    progress("Retrieving current SPX Index members...")
    members = request_frame(bq, bq.univ.members(INDEX), {"Name": bq.data.name()})
    if members.empty:
        raise ValueError("Bloomberg returned no current index constituents.")
    members = members.sort_values("ID").set_index("ID")
    if max_stocks is not None:
        members = members.head(max_stocks)
    tickers = members.index.tolist()
    end = market_date()
    start = end - pd.Timedelta(days=HISTORY_DAYS)
    dates = bq.func.range(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    spot_parts = []
    for i in range(0, len(tickers), BATCH_SIZE):
        progress(f"Underlying history: {min(i+BATCH_SIZE, len(tickers))}/{len(tickers)} stocks")
        spot_parts.append(request_frame(bq, tickers[i:i+BATCH_SIZE],
            {"spot": bq.data.px_last(dates=dates, fill="NA")}, history=True))
    spots = pd.concat(spot_parts, ignore_index=True)
    spots = spots.loc[spots["spot"].gt(0) & spots["DATE"].between(start, end)].copy()
    if spots.empty:
        raise ValueError("No valid underlying history was returned.")
    asof = pd.Timestamp(spots["DATE"].max())
    current_spot = spots.loc[spots["DATE"].eq(asof)].set_index("ID")["spot"]
    meta = members.rename_axis("Ticker").copy()
    meta["Selection status"] = "Not processed"
    errors, pairs = [], []
    chain_items = {"Expiry": bq.data.expire_dt(), "Strike": bq.data.strike_px(), "Side": bq.data.put_call()}
    completed, failures = 0, 0
    for i, ticker in enumerate(tickers, 1):
        progress(f"Selecting current call/put pairs: {i}/{len(tickers)} — {ticker}")
        try:
            if ticker not in current_spot:
                raise ValueError("No underlying price on the common as-of date.")
            try:
                # This already worked in the previous notebook. Only metadata is requested
                # for the chain; expensive historical prices are fetched for selected IDs.
                chain = request_frame(bq, bq.univ.options(ticker), chain_items)
            except Exception:
                failures += 1
                raise
            completed += 1
            chosen = select_pair(chain, float(current_spot[ticker]), asof, target_dte, tolerance)
            pairs.append({"Ticker": ticker, **chosen})
            meta.loc[ticker, "Selection status"] = "OK"
        except Exception as exc:
            meta.loc[ticker, "Selection status"] = str(exc)
            errors.append({"Stage": "Option selection", "Security": ticker, "Error": str(exc)})
            if completed == 0 and failures >= 3:
                raise RuntimeError("The first option-chain requests failed. " + str(exc)) from exc
    if pairs:
        meta = meta.join(pd.DataFrame(pairs).set_index("Ticker"))
    else:
        for col in ["Expiry", "Strike", "DTE", "Call contract", "Put contract"]:
            meta[col] = np.nan
    ids = sorted(set(meta["Call contract"].dropna()) | set(meta["Put contract"].dropna()))
    prices, price_errors = fetch_px_last(bq, ids, start, asof, progress)
    errors.extend(price_errors)
    coverage = history_coverage(prices, ids, start=start, end=asof) if ids else pd.DataFrame()
    by_id = {key: group for key, group in prices.groupby("ID")}
    by_stock = {key: group for key, group in spots.groupby("ID")}
    rows = []
    for ticker in tickers:
        p = by_stock.get(ticker, pd.DataFrame(columns=["DATE", "spot"]))
        panel = p[["DATE", "spot"]].set_index("DATE")
        for side, col in [("Call", "call_px"), ("Put", "put_px")]:
            sid = meta.loc[ticker, f"{side} contract"]
            q = by_id.get(sid, pd.DataFrame(columns=["DATE", "PX_LAST"]))
            # Every date is retained. No iloc[-1], tail(1), or drop_duplicates('ID').
            panel = panel.join(q.set_index("DATE")["PX_LAST"].rename(col), how="outer", validate="one_to_one")
            valid = q.loc[q["PX_LAST"].notna(), "DATE"]
            meta.loc[ticker, f"{side} first date"] = valid.min() if len(valid) else pd.NaT
            meta.loc[ticker, f"{side} last date"] = valid.max() if len(valid) else pd.NaT
            meta.loc[ticker, f"{side} valid dates"] = len(valid)
            meta.loc[ticker, f"{side} returned dates"] = q["DATE"].nunique()
        panel = panel.loc[(panel.index >= start) & (panel.index <= asof)]
        panel = panel.reindex(panel.index.union(pd.DatetimeIndex([asof])))
        panel.index.name = "date"
        panel["ticker"] = ticker
        for col in ["spot", "call_px", "put_px"]:
            panel[col] = pd.to_numeric(panel[col], errors="raise").astype(float)
        rows.append(panel.reset_index())
    raw = validate_history(pd.concat(rows, ignore_index=True))
    return {"history": raw, "contracts": meta, "errors": pd.DataFrame(errors),
            "coverage": coverage, "option_prices": prices, "asof": asof,
            "target_dte": target_dte, "tolerance": tolerance, "series_kind": "Same contract", "price_field": "PX_LAST"}




## 3. Optional daily reselection from verified historical chain snapshots
This is a separate data requirement, not a hidden API assumption. The expected schema is documented after the dashboard.

In [ ]:

def select_rolled_pairs(chain_snapshots: pd.DataFrame, spots: pd.DataFrame,
                        target_dte=TARGET_DTE, tolerance=EXPIRY_TOLERANCE_DAYS) -> pd.DataFrame:
    """Select one matched pair per date/underlying from REAL dated chain snapshots.

    chain_snapshots columns: date, ticker, ID, Expiry, Strike, Side.
    spots columns: DATE, ID (underlying), spot.
    Snapshot dates must mean the date the contract was in the historical chain,
    not its expiry or today's retrieval date. Supply standard-deliverable contracts
    and the complete eligible expiry/strike neighborhood. Completeness/provenance
    cannot be established from this schema alone.

    Select by expiry distance then absolute strike distance BEFORE looking at
    option prices. Missing quotes do not cause a different contract to be picked.
    This is nearest-listed-maturity/nearest-strike ATM, not interpolation to exact
    constant maturity or a self-financing strategy return series.
    """
    required = {"date", "ticker", "ID", "Expiry", "Strike", "Side"}
    if not isinstance(chain_snapshots, pd.DataFrame) or chain_snapshots.empty:
        raise ValueError("Real historical option-chain snapshots are required; today's chain is not a substitute.")
    if required.difference(chain_snapshots.columns):
        raise ValueError("Snapshot columns required: " + ", ".join(sorted(required)))
    c = chain_snapshots[list(required)].copy()
    c["date"] = pd.to_datetime(c["date"], errors="raise")
    c["Expiry"] = pd.to_datetime(c["Expiry"], errors="raise")
    for col in ["date", "Expiry"]:
        if c[col].isna().any() or c[col].dt.tz is not None:
            raise ValueError(f"{col}: use nonmissing, timezone-naive market dates")
        c[col] = c[col].dt.normalize()
    if c[["ticker", "ID"]].isna().any().any():
        raise ValueError("Snapshot ticker/ID cannot be missing")
    c["ticker"], c["ID"] = c["ticker"].astype(str), c["ID"].astype(str)
    if c.duplicated(["date", "ticker", "ID"]).any():
        raise ValueError("Duplicate date/ticker/contract rows in snapshots")
    c["Strike"] = pd.to_numeric(c["Strike"], errors="raise")
    c["Side"] = c["Side"].astype(str).str.upper().str.strip().map({"C": "C", "CALL": "C", "P": "P", "PUT": "P"})
    # An identifier cannot silently change expiry, strike or call/put side.
    invariant = c.groupby("ID")[["Expiry", "Strike", "Side"]].nunique(dropna=False)
    if invariant.gt(1).any().any():
        raise ValueError("A contract ID has inconsistent expiry/strike/side across snapshots")
    c["DTE"] = (c["Expiry"] - c["date"]).dt.days
    c = c.loc[c["Side"].notna() & c["Strike"].gt(0) & c["DTE"].ge(MIN_DTE)
              & c["DTE"].sub(target_dte).abs().le(tolerance)].copy()
    keys = ["date", "ticker", "Expiry", "Strike", "DTE"]
    # Different IDs on the same key/side can represent adjusted or duplicate contracts.
    # Do not silently choose one; remove the ambiguous candidates.
    c = c.loc[~c.duplicated(keys + ["Side"], keep=False)]
    empty_columns = ["date", "ticker", "Expiry", "Strike", "DTE", "Call contract", "Put contract"]
    if c.empty:
        return pd.DataFrame(columns=empty_columns)
    pairs = c.pivot(index=keys, columns="Side", values="ID")
    if not {"C", "P"}.issubset(pairs.columns):
        return pd.DataFrame(columns=empty_columns)
    pairs = pairs.dropna(subset=["C", "P"]).reset_index().rename(columns={"C": "Call contract", "P": "Put contract"})
    s = spots[["DATE", "ID", "spot"]].rename(columns={"DATE": "date", "ID": "ticker"})
    if s.duplicated(["date", "ticker"]).any():
        raise ValueError("Duplicate underlying/date observations")
    pairs = pairs.merge(s, on=["date", "ticker"], how="inner", validate="many_to_one")
    pairs = pairs.loc[np.isfinite(pairs["spot"]) & pairs["spot"].gt(0)].copy()
    pairs["expiry_distance"] = pairs["DTE"].sub(target_dte).abs()
    pairs["strike_distance"] = pairs["Strike"].sub(pairs["spot"]).abs()
    pairs = pairs.sort_values(["date", "ticker", "expiry_distance", "Expiry", "strike_distance", "Strike"])
    return pairs.drop_duplicates(["date", "ticker"], keep="first")[empty_columns].reset_index(drop=True)


def fetch_rolled_atm_history(bq, chain_snapshots: pd.DataFrame,
                             progress=lambda _: None, target_dte=TARGET_DTE,
                             tolerance=EXPIRY_TOLERANCE_DAYS,
                             max_stocks=None) -> dict:
    """BQNT price retrieval + daily reselection, given verified dated chain snapshots.

    No invented backdated options() parameter, generated expired tickers, or Excel
    automation. All spot/option prices are retrieved through BQL. Contract IDs
    must come from the user's actual Bloomberg historical snapshots, including
    the latest date. Missing snapshot dates stay missing, not carried forward.
    Uses today's SPX member universe (not a point-in-time index backtest).
    """
    if not isinstance(chain_snapshots, pd.DataFrame) or chain_snapshots.empty:
        raise ValueError("Supply actual historical chain snapshots, including expired contracts and the latest date.")
    if target_dte < MIN_DTE or tolerance < 0:
        raise ValueError("Invalid target DTE/tolerance")
    end = market_date()
    start = end - pd.Timedelta(days=HISTORY_DAYS)
    members = request_frame(bq, bq.univ.members(INDEX), {"Name": bq.data.name()}).sort_values("ID").set_index("ID")
    if max_stocks is not None:
        members = members.head(max_stocks)
    if members.empty:
        raise ValueError("No current index members")
    tickers = members.index.tolist()
    dates = bq.func.range(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    parts = []
    for i in range(0, len(tickers), BATCH_SIZE):
        progress(f"Rolled series: underlying history {min(i+BATCH_SIZE,len(tickers))}/{len(tickers)}")
        parts.append(request_frame(bq, tickers[i:i+BATCH_SIZE],
            {"spot": bq.data.px_last(dates=dates, fill="NA")}, history=True))
    spots = pd.concat(parts, ignore_index=True)
    spots = spots.loc[spots["DATE"].between(start, end) & spots["spot"].gt(0)].copy()
    if spots.empty:
        raise ValueError("No underlying history")
    asof = spots["DATE"].max()
    progress("Selecting matched expiry/strike pairs independently on each snapshot date...")
    selected = select_rolled_pairs(chain_snapshots, spots, target_dte, tolerance)
    if selected.empty:
        raise ValueError("No eligible pairs matched the underlying IDs/dates in the supplied snapshots.")
    ids = sorted(set(selected["Call contract"]) | set(selected["Put contract"]))
    prices, errors = fetch_px_last(bq, ids, start, asof, progress)
    chosen_prices = selected.copy()
    for side, col in [("Call", "call_px"), ("Put", "put_px")]:
        q = prices.rename(columns={"ID": f"{side} contract", "DATE": "date", "PX_LAST": col})
        chosen_prices = chosen_prices.merge(q, on=["date", f"{side} contract"], how="left", validate="many_to_one")
    base = spots.rename(columns={"ID": "ticker", "DATE": "date"})[["date", "ticker", "spot"]]
    raw = base.merge(chosen_prices[["date", "ticker", "call_px", "put_px"]],
                     on=["date", "ticker"], how="left", validate="one_to_one")
    # Ensure stale/missing names remain present in the latest dashboard.
    latest_keys = pd.DataFrame({"date": asof, "ticker": tickers})
    missing = latest_keys.merge(raw[["date", "ticker"]], on=["date", "ticker"], how="left", indicator=True)
    missing = missing.loc[missing["_merge"].eq("left_only"), ["date", "ticker"]]
    if not missing.empty:
        raw = pd.concat([raw, missing], ignore_index=True)
    raw = validate_history(raw)
    meta = members.rename_axis("Ticker").join(selected.loc[selected["date"].eq(asof)]
        .drop(columns="date").set_index("ticker"))
    meta["Selection status"] = np.where(meta["Call contract"].notna(), "OK", "No eligible dated snapshot/pair at as-of")
    for side, col in [("Call", "call_px"), ("Put", "put_px")]:
        valid = raw.loc[raw[col].notna()].groupby("ticker")["date"].agg(["min", "max", "count"])
        meta[f"{side} first date"] = valid["min"]
        meta[f"{side} last date"] = valid["max"]
        meta[f"{side} valid dates"] = valid["count"].reindex(meta.index).fillna(0)
        # Counts refer to selected dated rows here, not one contract's entire history.
        available = chosen_prices.groupby("ticker")["date"].nunique()
        meta[f"{side} returned dates"] = available.reindex(meta.index).fillna(0)
    return {"history": raw, "contracts": meta, "errors": pd.DataFrame(errors),
            "coverage": history_coverage(prices, ids, start=start, end=asof),
            "option_prices": prices, "selected_by_date": selected, "asof": asof,
            "target_dte": target_dte, "tolerance": tolerance,
            "series_kind": "Daily reselected ATM (nearest listed maturity)", "price_field": "PX_LAST"}



## 4. Dashboard definitions

In [ ]:

def launch_dashboard(bq=None, chain_snapshots=None):
    """Create notebook widgets; Load is the only Bloomberg retrieval trigger.

    Without snapshots: each currently chosen contract's own PX_LAST history.
    With actual chain_snapshots: daily reselected ATM history, including expired IDs.
    """
    import bql
    import ipywidgets as w
    import matplotlib.pyplot as plt
    from IPython.display import HTML, display
    bq = bq if bq is not None else bql.Service()
    rolled = chain_snapshots is not None
    series_label = "Daily reselected ATM" if rolled else "Same-contract"
    target = w.IntText(value=TARGET_DTE, description="Target DTE:")
    band = w.IntText(value=EXPIRY_TOLERANCE_DAYS, description="DTE band ±:")
    full_window = w.Checkbox(value=REQUIRE_FULL_WINDOW, description="Require full elapsed year")
    min_obs = w.IntText(value=MIN_OBSERVATIONS, description="Min prior obs:")
    basis = w.Dropdown(options=["Raw premium", "Premium / spot"], description="Basis:")
    metric = w.Dropdown(options=["Call Z", "Put Z", "Spread Z"], value="Call Z", description="Rank by:")
    minimum = w.FloatSlider(value=0, min=0, max=4, step=0.5, description="Min |Z|:", continuous_update=False)
    top_n = w.Dropdown(options=[20, 50, 100, 600], value=50, description="Max rows:")
    search = w.Text(placeholder="Ticker contains...", description="Search:", continuous_update=False)
    stock = w.Dropdown(options=[], description="Chart:", layout=w.Layout(width="330px"))
    refresh = w.Button(description="Load / refresh Bloomberg", icon="refresh", layout=w.Layout(width="240px"))
    progress = w.HTML()
    banner = w.HTML(value=(
        "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
        "<p>Click <b>Load / refresh Bloomberg</b>. No demo or CSV data path.<br>"
        + ("Dated historical snapshots supplied: contracts will be reselected independently each date.</p>"
           if rolled else "Each selected contract is compared with its own history. This is NOT a historically rolled ATM series.</p>")
    ))
    table_out, chart_out, error_out = w.Output(), w.Output(), w.Output()
    state = {"data": None, "models": {}, "loading": False}

    def model():
        key = (basis.value, int(min_obs.value), bool(full_window.value))
        if key not in state["models"]:
            state["models"][key] = compute_dashboard(
                state["data"]["history"], basis.value, WINDOW_DAYS, int(min_obs.value),
                asof=state["data"]["asof"], require_full_window=bool(full_window.value),
            )
        return state["models"][key]

    def draw_table(*_):
        if state["loading"] or state["data"] is None:
            return
        m = model()
        key = METRICS[metric.value]
        all_rows = m["latest"].join(state["data"]["contracts"])
        for label, k in METRICS.items():
            stem = label.split()[0]
            all_rows[f"{stem} prior obs"] = m["counts"][k].loc[m["asof"]].fillna(0).astype(int)
            all_rows[f"{stem} Z status"] = m["statuses"][k]
        all_rows["Z status"] = m["statuses"][key]
        bad_selection = all_rows["Selection status"].ne("OK")
        all_rows.loc[bad_selection, "Z status"] = all_rows.loc[bad_selection, "Selection status"]
        t = all_rows.copy()
        if search.value.strip():
            t = t.loc[t.index.str.contains(search.value.strip(), case=False, regex=False)]
        if minimum.value > 0:
            t = t.loc[t[metric.value].abs().ge(minimum.value)]
        t = t.assign(_rank=t[metric.value].abs()).sort_values("_rank", ascending=False, na_position="last").head(top_n.value)
        columns = ["Name", "Spot", "Expiry", "DTE", "Strike", "Call PX_LAST", "Put PX_LAST",
                   "Call Z", "Put Z", "Spread Z", "Call prior obs", "Put prior obs", "Spread prior obs",
                   "Call Z status", "Put Z status", "Z status",
                   "Call first date", "Put first date", "Call last date", "Put last date",
                   "Call valid dates", "Put valid dates", "Call returned dates", "Put returned dates",
                   "Call contract", "Put contract"]
        t = t.reindex(columns=columns)
        for col in ["Expiry", "Call first date", "Put first date", "Call last date", "Put last date"]:
            t[col] = pd.to_datetime(t[col]).dt.strftime("%Y-%m-%d").fillna("—")
        with table_out:
            table_out.clear_output(wait=True)
            display(HTML("<p><b>" + ("Full-year eligibility required" if full_window.value else "AVAILABLE-HISTORY SCORES — not full-year coverage") + f"</b>; min prior obs = {int(min_obs.value)}. All option values: PX_LAST, not midpoint.</p>"))
            display(HTML(f"<p><b>{all_rows[metric.value].notna().sum()}/{len(all_rows)} stocks have a valid {escape(metric.value)}.</b> "
                         "Missing latest prices are not carried forward. Z-score ranking is by absolute value.</p>"))
            display(HTML('<div style="overflow-x:auto;max-height:520px;overflow-y:auto">'
                         + t.to_html(float_format=lambda v: f"{v:,.2f}", na_rep="—", escape=True) + "</div>"))

    def draw_chart(*_):
        if state["loading"] or state["data"] is None or not stock.value:
            return
        m, ticker = model(), stock.value
        left = m["asof"] - pd.Timedelta(days=365)
        info = state["data"]["contracts"].loc[[ticker]]
        with chart_out:
            chart_out.clear_output(wait=True)
            display(HTML(info[["Call contract", "Put contract", "Selection status"]].to_html(escape=True)))
            fig, ax = plt.subplots(figsize=(11, 3.2))
            for label, key in [("Call Z", "call"), ("Put Z", "put")]:
                z = m["scores"][key].loc[left:, ticker]
                ax.plot(z.index, z.values, label=label)
            ax.axhline(2, linestyle="--", linewidth=0.8)
            ax.axhline(-2, linestyle="--", linewidth=0.8)
            ax.axhline(0, linestyle=":", linewidth=0.8)
            label = "Full-year required" if full_window.value else "Available history"
            ax.set_title(f"{ticker} | {series_label} 365-day window | {label} | {basis.value}")
            ax.set_ylabel("Standard deviations")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(11, 3.0))
            for label, key in [("Call PX_LAST", "call_px"), ("Put PX_LAST", "put_px")]:
                p = m["price"][key].loc[left:, ticker]
                ax.plot(p.index, p.values, label=label)
            ax.set_title(f"{ticker} | {series_label}: historical PX_LAST")
            ax.set_ylabel("PX_LAST (USD premium units)")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)

    def refresh_data(*_):
        if int(min_obs.value) < 2:
            progress.value = "Min prior observations must be at least 2."
            return
        state["loading"] = True
        state["data"], state["models"] = None, {}
        refresh.disabled = target.disabled = band.disabled = True
        table_out.clear_output()
        chart_out.clear_output()
        error_out.clear_output()
        try:
            loader = fetch_rolled_atm_history if rolled else fetch_bloomberg_history
            kwargs = {"chain_snapshots": chain_snapshots} if rolled else {}
            data = loader(bq, progress=lambda text: setattr(progress, "value", escape(text)),
                          target_dte=int(target.value), tolerance=int(band.value),
                          max_stocks=MAX_STOCKS, **kwargs)
            state["data"] = data
            tickers = sorted(data["history"]["ticker"].unique())
            old = stock.value
            stock.options = tickers
            stock.value = old if old in tickers else tickers[0]
            loaded = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")
            banner.value = (
                "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
                f"<p><b>Bloomberg BQL data | Common as-of date: {data['asof']:%Y-%m-%d}</b><br>"
                f"Loaded {loaded}. Latest available daily observations; NOT a streaming feed.<br>"
                f"{escape(data['series_kind'])}: target {data['target_dte']} DTE ± {data['tolerance']} days, "
                "closest eligible expiry then closest strike to spot; identical expiry/strike for call and put.<br>"
                f"Baseline: previous {WINDOW_DAYS} calendar days, excluding the current date; "
                "coverage and minimum-observation controls below determine eligibility.<br>"
                + ("<b>Daily-reselected nearest-listed ATM series; not exact constant maturity.</b> "
                   if rolled else "<b>Same-contract history, NOT historically rolled ATM.</b> Recent contracts can lack a full year. ")
                + "All option values use <b>PX_LAST</b>, not bid/ask midpoints. Returned dates do not establish trade freshness. "
                "Check quote timing, deliverables and corporate actions before relying on this prototype.</p>"
            )
            counts = data["coverage"]["Valid prices"] if not data["coverage"].empty else pd.Series(dtype=float)
            single = int(counts.le(1).sum())
            progress.value = (f"Loaded {len(tickers)} stock rows; {len(data['errors'])} query/selection issues. "
                              f"{single}/{len(counts)} contracts have at most one valid PX_LAST date. "
                              "See raw history coverage below.")
            with error_out:
                if not data["coverage"].empty:
                    display(HTML("<h4>Raw PX_LAST history coverage</h4>" + data["coverage"].to_html(escape=True)))
            if not data["errors"].empty:
                with error_out:
                    display(HTML(data["errors"].to_html(index=False, escape=True)))
        except Exception as exc:
            banner.value = "<h3>Bloomberg loading failed</h3><p>" + escape(str(exc)) + "</p>"
            progress.value = "No substitute data was loaded."
        finally:
            state["loading"] = False
            refresh.disabled = target.disabled = band.disabled = False
        if state["data"] is not None:
            try:
                draw_table()
                draw_chart()
            except Exception as exc:
                with error_out:
                    display(HTML("<b>Rendering/calculation error:</b> " + escape(str(exc))))

    for control in [metric, minimum, top_n, search]:
        control.observe(draw_table, names="value")
    stock.observe(draw_chart, names="value")
    def redraw(*_):
        try:
            draw_table()
            draw_chart()
        except Exception as exc:
            progress.value = escape(str(exc))
    for control in [basis, min_obs, full_window]:
        control.observe(redraw, names="value")
    refresh.on_click(refresh_data)
    errors_panel = w.Accordion(children=[error_out])
    errors_panel.set_title(0, "Historical date coverage / query issues")
    errors_panel.selected_index = None
    controls = w.Box([basis, metric, minimum, top_n, search], layout=w.Layout(display="flex", flex_flow="row wrap"))
    dashboard = w.VBox([banner, w.HBox([target, band, refresh]), progress,
                        w.HBox([full_window, min_obs]), controls,
                        table_out, stock, chart_out, errors_panel])
    dashboard.data_state = state  # Inspect history/option_prices/coverage after Load.
    return dashboard


def main():
    from IPython.display import display
    dashboard = launch_dashboard()
    display(dashboard)
    return dashboard



## 5. Test the two AAPL identifiers from your output first

This requests one year for **only two known contract IDs**. `Valid prices` counts nonnegative finite price observations; `Returned dates` can include missing prices. The first/last valid dates refer to Bloomberg observation dates, not verified transaction timestamps.

Set `compare_bid_ask=True` to compare the old fields with the new field. A field failure is reported separately and does not erase successful results from the other fields.

In [ ]:
import bql
from IPython.display import display

bq = bql.Service()

probe = probe_option_history(
    bq,
    ["BBG025H8SHM8", "BBG025H8T570"],  # Exact call/put IDs from your AAPL row
    compare_bid_ask=True,
)
display(probe["coverage"])
if not probe["errors"].empty:
    display(probe["errors"])
if "PX_LAST" in probe["frames"]:
    px_last_check = probe["frames"]["PX_LAST"]
    display(px_last_check.sort_values(["ID", "DATE"]).groupby("ID").tail(10))


## 6. Load the dashboard

`MAX_STOCKS = None` requests every current S&P 500 constituent. Set it to `5` for an initial small-scope load. Click **Load / refresh Bloomberg** after displaying the dashboard.

Keep **Require full elapsed year** checked for full-year eligibility. Deliberately unchecking it changes the statistic to available-history coverage within the same 365-day maximum window. It does not create missing data. Both the minimum count and the chosen coverage policy appear above the table.

In [ ]:
MAX_STOCKS = None        # All current S&P 500 constituents; 5 for a smaller test
TARGET_DTE = 30          # Set to 90 for a roughly three-month maturity target
EXPIRY_TOLERANCE_DAYS = 14
MIN_OBSERVATIONS = 200
REQUIRE_FULL_WINDOW = True

dashboard = launch_dashboard(bq)
display(dashboard)


## 7. Inspect loaded dates directly

This cell reads the data already in the dashboard; it does not make another Bloomberg request. Contract-level coverage is also in the **Historical date coverage / query issues** panel.

In [ ]:
loaded = dashboard.data_state["data"]
if loaded is None:
    print("Click Load / refresh Bloomberg, then rerun this inspection cell.")
else:
    display(loaded["coverage"])
    display(loaded["option_prices"].sort_values(["ID", "DATE"]).groupby("ID").tail(5))
    display(loaded["history"].groupby("ticker").agg(
        first_panel_date=("date", "min"),
        last_panel_date=("date", "max"),
        call_valid_dates=("call_px", "count"),
        put_valid_dates=("put_px", "count"),
    ))


## 8. The genuine daily rolled-ATM path

The Bloomberg guidance supplied in the conversation says historical chain snapshots must be obtained separately (for example through a supported Terminal/Excel historical-chain workflow). This notebook does **not** independently verify or automate that workflow.

Supply an existing DataFrame named `chain_snapshots` with these exact columns:

| Column | Meaning |
|---|---|
| `date` | Date on which this contract was in the historical chain, as a timezone-naive market-date label. |
| `ticker` | Bloomberg underlying ID matching the `SPX Index` member/spot IDs used here. |
| `ID` | Real Bloomberg option identifier, including expired contracts where relevant. |
| `Expiry` | Contract expiry date. |
| `Strike` | Strike price in consistent units. |
| `Side` | `C`/`P` or `CALL`/`PUT`. |

Supply the eligible expiry/strike neighborhood for each date, including the latest date. Do not copy today's chain onto historical dates. A schema alone cannot prove point-in-time completeness, deliverable standardization, or provenance.

The optional loader filters each dated chain by target days-to-expiry, chooses the nearest listed expiry and then the nearest strike with both call and put, and fetches those selected IDs through BQL. **Selection happens before inspecting option prices.** Missing selected prices or missing snapshot dates remain missing; the algorithm does not substitute a different option because its price happens to be available.

A nearest-listed maturity series is an approximation, not an exactly constant-maturity interpolated series. It is a premium-level series, not a self-financing roll-strategy return. Current constituents are used throughout; this is not a point-in-time index-membership backtest.

One year of snapshots supports a current one-year statistic when coverage is sufficient; roughly two years are needed to display a full year of rolling one-year z-scores. Simply appending four quarterly contract histories does not define one value per day, constant maturity, or daily ATM selection. The sample's September 18, 2026 expiry also precedes September 29, 2026 and cannot supply the latter day's current option observation.

In [ ]:
# Only activates when you have supplied actual dated Bloomberg chain records.
if "chain_snapshots" in globals():
    rolled_dashboard = launch_dashboard(bq, chain_snapshots=chain_snapshots)
    display(rolled_dashboard)
else:
    print("No chain_snapshots DataFrame supplied. The default dashboard remains explicitly same-contract.")


## Method and verification notes

The z-score at date t is `(price_t - prior_window_mean) / prior_window_sample_std`, with the current observation excluded. A missing current price, too few valid observations, insufficient elapsed coverage when required, or zero standard deviation produces no score. The time-based window does not extend backward to replace missing observations.

The raw premium comparison is affected by changing moneyness, remaining maturity, corporate actions, and market conditions. An unusually high z-score is not by itself evidence of option mispricing. `PX_LAST` observation dates do not guarantee recent trading, simultaneous call/put observations, or executable prices. Midpoints and last prices are different price conventions.

**Sources and implementation basis:** BQL `options()`, `px_last(dates=range(...))`, and historical-chain constraints follow the Bloomberg guidance pasted by the user in this conversation; they were not independently live-tested. Date-window semantics and keyed joins are documented in [pandas rolling](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) and [pandas merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).

**Offline validation:** Nine data-processing/calculation tests passed, including raw-vs-valid date counts, preservation of contract/date rows, exclusion of the current observation, missing-date behavior, separate full/partial coverage policies, duplicate-date rejection, and reselection against each date's spot. A widget-initialization test was skipped because the local test environment does not have ipywidgets. Bloomberg entitlements, vendor data availability, actual BQL execution, and BQNT front-end rendering still require your session.
